In [ ]:
import pandas as pd
from fastparquet import write
from fastparquet import ParquetFile
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
import numpy as np
import glob
import os

In [10]:
data_pipeline = "scale"
input_pipeline = "impute"

In [11]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [12]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [13]:
cat_cols = ["gender"]
ordinal_cols = ["stress_level"]
binary_cols = ["academic_work_impact"]

In [14]:
log_cols = ["social_media_hours", "gaming_hours", "work_study_hours", "daily_extra_screen_time_hours", "phone_activity", "hourly_notifications_per_day", "hourly_app_opens_per_day"]
std_cols = ["total_activity", "work_ratio", "gaming_ratio", "social_media_ratio", "screen_sleep_ratio", "daily_screen_time_hours", "weekend_screen_time", "daily_free_hours", "weekend_extra_screen_time", "stress_daily_hours_impact", "stress_social_media_hours_impact", "stress_work_study_hours_impact"]
minmax_cols = ["age", "sleep_hours", "notifications_per_day", "app_opens_per_day", "stress_level", "stress_app_opens_impact", "stress_notifications_impact"]
len_norm_cols = len(log_cols) + len(std_cols) + len(minmax_cols)
len_norm_cols

26

In [15]:
ohe_dir = os.path.join(data_path, input_pipeline)
train_files = glob.glob(os.path.join(ohe_dir, "train_*.parq"))
test_files = glob.glob(os.path.join(ohe_dir, "test_*.parq"))
y = pd.read_csv(Path(ohe_dir) / '../raw/train.csv')[target_column]

In [17]:
for X_file, X_test_file in zip(train_files, test_files):
    X = ParquetFile(X_file).to_pandas()
    X_test = ParquetFile(X_test_file).to_pandas()

    bin_cols = binary_cols + (X.select_dtypes(include="bool").columns.to_list())

    for col in bin_cols:
        if col not in X:
            continue
        X[col] = X[col].astype(int)
        X_test[col] = X_test[col].astype(int)

    for col in log_cols:
        if col not in X:
            continue
        X[col] = np.log1p(X[col])
        X_test[col] = np.log1p(X_test[col])

    scale_cols = std_cols + log_cols

    X_cv_scaled = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)

    kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)

    sc_cols = list(set(X.columns) & set(scale_cols))

    for train_index, valid_index in kf.split(X, y):
        X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
        y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train[sc_cols])
        X_valid_scaled = scaler.transform(X_valid[sc_cols])
        
        X_valid_full = X_valid.copy()
        X_valid_full[sc_cols] = X_valid_scaled
        
        X_cv_scaled.loc[valid_index] = X_valid_full

    scaler = StandardScaler()
    scaler.fit(X[sc_cols])

    X_test_scaled = scaler.transform(X_test[sc_cols])

    X_test_full = X_test.copy()
    X_test_full[sc_cols] = X_test_scaled

    X = X_cv_scaled.copy()
    X_test = X_test_full.copy()

    X_cv_scaled = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)

    kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)

    mimx_cols = list(set(X.columns) & set(minmax_cols))

    for train_index, valid_index in kf.split(X, y):
        X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
        y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
        
        scaler = MinMaxScaler()
        X_train_scaled = scaler.fit_transform(X_train[mimx_cols])
        X_valid_scaled = scaler.transform(X_valid[mimx_cols])
        
        X_valid_full = X_valid.copy()
        X_valid_full[mimx_cols] = X_valid_scaled
        
        X_cv_scaled.loc[valid_index] = X_valid_full

    scaler = MinMaxScaler()
    scaler.fit(X[mimx_cols])

    X_test_scaled = scaler.transform(X_test[mimx_cols])

    X_test_full = X_test.copy()
    X_test_full[mimx_cols] = X_test_scaled

    X = X_cv_scaled.copy()
    X_test = X_test_full.copy()

    out_path = Path(data_path) / f"{data_pipeline}"
    out_path.mkdir(parents=True, exist_ok=True)

    write(out_path / Path(X_file).name, X)
    write(out_path / Path(X_test_file).name, X_test)